# 09 - Gold Business Tables

## Objective

The objective of this notebook is to create business-ready Gold tables from the validated Fact and Dimension tables.

The Gold layer is designed for business analysis and reporting. Instead of simply copying Silver or Fact tables, these tables organize data around specific business questions.

## Source Tables

The Gold tables are created using the validated analytical model:

- `fact_sales`
- `fact_order_payment`
- `dim_customer`
- `dim_product`
- `dim_order`
- `dim_payment`
- `dim_date`

## Gold Business Tables

The following business-oriented tables will be created:

### 1. `gold_sales_summary`

Provides overall and time-based sales metrics such as:

- Total sales
- Total product sales
- Total freight
- Number of orders
- Number of products sold
- Average order value
- Sales by date/month/year

### 2. `gold_customer_analysis`

Provides customer-level business metrics such as:

- Customer location
- Number of orders
- Total spending
- Average order value
- Customer purchase frequency

### 3. `gold_product_analysis`

Provides product-level performance metrics such as:

- Product category
- Number of orders
- Quantity/order-line count
- Total sales
- Average price
- Total freight
- Product performance

### 4. `gold_payment_analysis`

Provides payment-related business metrics such as:

- Payment type
- Transaction count
- Total payment value
- Average payment value
- Payment installments

### 5. `gold_delivery_analysis`

Provides delivery performance metrics such as:

- Order status
- Order purchase date
- Estimated delivery date
- Actual customer delivery date
- Delivery duration
- Delivery delay
- On-time vs delayed delivery

## Business Purpose

The Gold layer converts the analytical data model into business-friendly datasets that can be used to answer questions such as:

- How much revenue is generated?
- How are sales changing over time?
- Which products and categories perform best?
- Which customers generate the most sales?
- Which payment methods are most frequently used?
- How well are orders being delivered?
- How many orders are delayed or cancelled?

## Data Flow

Raw Tables
↓
Data Quality
↓
Data Exploration
↓
Clean/Silver Tables
↓
Dimensions + Fact Tables
↓
**Gold Business Tables**
↓
Advanced SQL Analysis
↓
Business Insights

## Validation

Each Gold table will be validated for:

- Row counts
- NULL values
- Duplicate records
- Correct aggregations
- Referential integrity
- Business-rule consistency
- Accuracy of calculated metrics

## Outcome

The Gold layer will provide clean, aggregated, business-oriented datasets for advanced SQL analysis, business insights, reporting, and downstream analytics.

#####Create gold_sales_summary

In [0]:
%sql

CREATE OR REPLACE TABLE gold_sales_summary AS
SELECT
    COUNT(*) AS total_order_items,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT product_id) AS total_products,

    ROUND(SUM(price), 2) AS total_product_sales,
    ROUND(SUM(freight_value), 2) AS total_freight,
    ROUND(SUM(price + freight_value), 2) AS total_sales_value,

    ROUND(
        SUM(price + freight_value) / COUNT(DISTINCT order_id),
        2
    ) AS average_order_value,

    ROUND(MIN(price), 2) AS minimum_product_price,
    ROUND(MAX(price), 2) AS maximum_product_price

FROM fact_sales;

In [0]:
%sql

SELECT *
FROM gold_sales_summary;

In [0]:
%sql

SELECT
    COUNT(*) AS summary_rows,
    SUM(total_sales_value) AS sales_value
FROM gold_sales_summary;

#####gold_customer_analysis

In [0]:
%sql

CREATE OR REPLACE TABLE gold_customer_analysis AS
SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(*) AS total_order_items,
    ROUND(SUM(price + freight_value), 2) AS total_spending,
    ROUND(AVG(price + freight_value), 2) AS average_order_item_value,
    ROUND(SUM(freight_value), 2) AS total_freight
FROM fact_sales
GROUP BY customer_id;

In [0]:
%sql

SELECT *
FROM gold_customer_analysis
LIMIT 10;

In [0]:
%sql

SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM gold_customer_analysis;

####Validating the difference

In [0]:
%sql

SELECT
    (SELECT COUNT(*) FROM dim_customer) AS all_customers,
    (SELECT COUNT(*) FROM gold_customer_analysis) AS customers_with_sales,
    (SELECT COUNT(*) FROM dim_customer)
      - (SELECT COUNT(*) FROM gold_customer_analysis) AS customers_without_sales;

In [0]:
%sql

SELECT *
FROM gold_customer_analysis
ORDER BY total_spending DESC
LIMIT 10;

#####gold_product_analysis

In [0]:
%sql
CREATE OR REPLACE TABLE gold_product_analysis AS

SELECT
    product_id,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(*) AS total_order_items,
    SUM(price) AS total_sales,
    AVG(price) AS average_price,
    SUM(freight_value) AS total_freight,
    SUM(price + freight_value) AS total_sales_value

FROM fact_sales

GROUP BY product_id;

In [0]:
%sql
SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT product_id) AS unique_products
FROM gold_product_analysis;

In [0]:
%sql
SELECT
    product_id,
    total_orders,
    total_order_items,
    total_sales,
    total_sales_value
FROM gold_product_analysis
ORDER BY total_sales_value DESC
LIMIT 10;

####Validate the complete product table

In [0]:
%sql
SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT product_id) AS unique_products
FROM gold_product_analysis;

In [0]:
%sql
SELECT
    COUNT(*) AS negative_sales,
    COUNT(*) AS null_products
FROM gold_product_analysis
WHERE total_sales < 0
   OR product_id IS NULL;

#####gold_payment_analysis

In [0]:
%sql

CREATE OR REPLACE TABLE gold_payment_analysis AS

SELECT
    payment_type,
    COUNT(*) AS transaction_count,
    SUM(payment_value) AS total_payment_value,
    AVG(payment_value) AS average_payment_value,
    MAX(payment_value) AS max_payment_value,
    MIN(payment_value) AS min_payment_value

FROM clean_order_payments

GROUP BY payment_type;

In [0]:
%sql

SELECT
    COUNT(*) AS payment_types,
    COUNT(DISTINCT payment_type) AS unique_payment_types
FROM gold_payment_analysis;

In [0]:
%sql

SELECT *
FROM gold_payment_analysis
ORDER BY total_payment_value DESC;

In [0]:
%sql

DESCRIBE clean_orders;

#####gold_delivery_analysis

In [0]:
%sql

CREATE OR REPLACE TABLE gold_delivery_analysis AS

SELECT
    COUNT(*) AS total_orders,

    COUNT(
        CASE
            WHEN order_status = 'delivered' THEN 1
        END
    ) AS delivered_orders,

    COUNT(
        CASE
            WHEN order_status = 'canceled' THEN 1
        END
    ) AS cancelled_orders,

    COUNT(
        CASE
            WHEN order_delivered_customer_date IS NOT NULL THEN 1
        END
    ) AS orders_with_delivery_date,

    ROUND(
        AVG(
            CASE
                WHEN order_delivered_customer_date IS NOT NULL
                THEN DATEDIFF(
                    order_delivered_customer_date,
                    order_purchase_timestamp
                )
            END
        ),
        2
    ) AS average_delivery_days,

    MIN(
        CASE
            WHEN order_delivered_customer_date IS NOT NULL
            THEN DATEDIFF(
                order_delivered_customer_date,
                order_purchase_timestamp
            )
        END
    ) AS minimum_delivery_days,

    MAX(
        CASE
            WHEN order_delivered_customer_date IS NOT NULL
            THEN DATEDIFF(
                order_delivered_customer_date,
                order_purchase_timestamp
            )
        END
    ) AS maximum_delivery_days

FROM clean_orders;

In [0]:
%sql

SELECT *
FROM gold_delivery_analysis;

In [0]:
%sql

SELECT
    total_orders,
    delivered_orders,
    cancelled_orders,
    orders_with_delivery_date,
    average_delivery_days,
    minimum_delivery_days,
    maximum_delivery_days
FROM gold_delivery_analysis;

In [0]:
%sql

SELECT
    COUNT(*) AS invalid_delivery_dates
FROM clean_orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_delivered_customer_date < order_purchase_timestamp;